In [1]:
import pandas as pd
import sqlite3

# ── LOAD DATA ──────────────────────────────────────────────────────────────────

df = pd.read_csv('CRIM_GEN_REG.csv')
iccs = pd.read_csv('ICCS.csv')
unit = pd.read_csv('UNIT.csv')

# ── FILTER: Country level only (2-letter codes), NR unit only ─────────────────

# Country codes are exactly 2 characters (AT, DE, FR...)
df_country = df[df['geo'].str.len() == 2].copy()

# Keep only raw numbers (NR), not per-inhabitant
df_country = df_country[df_country['unit'] == 'NR'].copy()

# ── ADD READABLE CRIME NAMES ───────────────────────────────────────────────────

crime_map = {
    'ICCS0101':    'Intentional Homicide',
    'ICCS02011':   'Rape',
    'ICCS0401':    'Robbery',
    'ICCS0501':    'Burglary',
    'ICCS05012':   'Residential Burglary',
    'ICCS0502':    'Theft',
    'ICCS050211':  'Motor Vehicle Theft',
}
df_country['crime_type'] = df_country['iccs'].map(crime_map)

# Add full country names
country_names = {
    'AL':'Albania','AT':'Austria','BA':'Bosnia','BE':'Belgium','BG':'Bulgaria',
    'CH':'Switzerland','CY':'Cyprus','CZ':'Czechia','DE':'Germany','DK':'Denmark',
    'EE':'Estonia','EL':'Greece','ES':'Spain','FI':'Finland','FR':'France',
    'HR':'Croatia','HU':'Hungary','IE':'Ireland','IS':'Iceland','IT':'Italy',
    'LI':'Liechtenstein','LT':'Lithuania','LU':'Luxembourg','LV':'Latvia',
    'ME':'Montenegro','MT':'Malta','NL':'Netherlands','NO':'Norway','PL':'Poland',
    'PT':'Portugal','RO':'Romania','RS':'Serbia','SE':'Sweden','SI':'Slovenia',
    'SK':'Slovakia','TR':'Türkiye','XK':'Kosovo',
}
df_country['country_name'] = df_country['geo'].map(country_names)

# ── FINAL CLEAN TABLE ──────────────────────────────────────────────────────────

df_clean = df_country[['geo', 'country_name', 'crime_type', 'TIME_PERIOD', 'value']].copy()
df_clean.columns = ['country_code', 'country', 'crime_type', 'year', 'count']
df_clean = df_clean.dropna(subset=['crime_type', 'country', 'count'])

print("Clean dataset shape:", df_clean.shape)
print("\nSample:")
print(df_clean.head(10))

# ── EXPORT TO SQLITE ───────────────────────────────────────────────────────────

conn = sqlite3.connect('eu_crime.db')
df_clean.to_sql('crimes', conn, if_exists='replace', index=False)
conn.close()

print("\n✅ eu_crime.db created successfully!")
print(f"   {len(df_clean):,} rows exported to 'crimes' table")

Clean dataset shape: (2452, 5)

Sample:
  country_code  country            crime_type  year  count
0           AL  Albania  Intentional Homicide  2008   88.0
1           AL  Albania  Intentional Homicide  2009   82.0
2           AL  Albania  Intentional Homicide  2010  118.0
3           AL  Albania  Intentional Homicide  2011  124.0
4           AL  Albania  Intentional Homicide  2012  126.0
5           AL  Albania  Intentional Homicide  2013  107.0
6           AL  Albania  Intentional Homicide  2014   98.0
7           AL  Albania  Intentional Homicide  2015   54.0
8           AL  Albania  Intentional Homicide  2016   71.0
9           AL  Albania  Intentional Homicide  2017   52.0

✅ eu_crime.db created successfully!
   2,452 rows exported to 'crimes' table
